# Inicialización

## Módulos

In [ ]:
# Módulos iternos
import os
from pathlib import Path

# Módulos de terceros (instalar por pip)
import numpy as np
# import matplotlib.pyplot as plt
from tqdm import tqdm

# Módulos propios
from ti_modules import information

# Simulaciones

## Compresión

#### Simulación de los límites de compresión de 5 archivos binarios, 3 generados por 3 fuentes distintas, uno aleatorio, y uno semialeatorio (3 bits por byte fijos, y 5 bits por byte generados con distribución uniforme)

In [ ]:
symbols = ["0", "1"]
pmf = np.array([0.1, 0.9])
memory_pmf = np.array([[0.1, 0.9], [0.3, 0.7], [0.4, 0.6], [0.8, 0.2]])
# memory_pmf = np.array([[0.75, 0.25], [0.95, 0.05], [0, 1], [0.5, 0.5]])  # distribución alternativa para probar

output_bytes = 8000*2**10  # 8000 kB
fixed_bits_per_byte = 3

source = information.Source((symbols, pmf), 8)
memory_source = information.MemorySource((symbols, memory_pmf), 8)
unconditional_source = memory_source.unconditional_source()

print(memory_source)

print(f"Entropía de la fuente base: {source.base_entropy:.3g} bits por símbolo del alfabeto")
print(f"Entropía de la fuente con memoria base: {memory_source.base_entropy:.3g} bits por símbolo del alfabeto")
print(f"Entropía de la fuente afín base: {unconditional_source.base_entropy/8:.3g} bits por símbolo del alfabeto")

current_dir_path = Path.cwd()
output_dir_path = current_dir_path.parent.joinpath("output")
if not output_dir_path.exists():
    os.mkdir(output_dir_path)  # crea la carpeta de output si no existe

source_output_path = output_dir_path.joinpath("compression_sim_source_output")
memory_source_output_path = output_dir_path.joinpath("compression_sim_memory_source_output")
unconditional_source_output_path = output_dir_path.joinpath("compression_sim_unconditional_source_output")
random_output_path = output_dir_path.joinpath("compression_sim_random_output")
semirandom_output_path = output_dir_path.joinpath("compression_sim_semirandom_output")

expected_source_output_compressed_size = output_bytes*source.base_entropy
expected_memory_source_output_compressed_size = output_bytes*memory_source.base_entropy
expected_unconditional_source_output_compressed_size = output_bytes*unconditional_source.base_entropy/8
mask = 2**(8 - fixed_bits_per_byte) - 1
expected_semirandom_output_compressed_size = output_bytes*(8 - fixed_bits_per_byte)/8

print(f"Tamaño de los archivos de salida: {output_bytes/1024:.0f} Kb")
print(f"Tamaño esperado del archivo generado por la fuente comprimido: {expected_source_output_compressed_size/1024:.0f} Kb")
print(f"Tamaño esperado del archivo generado por la fuente con memoria comprimido: {expected_memory_source_output_compressed_size/1024:.0f} Kb")
print(f"Tamaño esperado del archivo generado por la fuente afín comprimido: {expected_unconditional_source_output_compressed_size/1024:.0f} Kb")
print(f"Tamaño esperado del archivo semialeatorio comprimido: {expected_semirandom_output_compressed_size/1024:.0f} Kb")

with open(source_output_path, "bw") as file:
    for _ in tqdm(range(output_bytes)):
        file.write(bytes([source.symbol_to_index(source())]))

with open(memory_source_output_path, "bw") as file:
    for _ in tqdm(range(output_bytes)):
        file.write(bytes([memory_source.symbol_to_index(memory_source())]))

with open(unconditional_source_output_path, "bw") as file:
    for _ in tqdm(range(output_bytes)):
        file.write(bytes([unconditional_source.symbol_to_index(unconditional_source())]))  

with open(random_output_path, "bw") as random_file:
    with open(semirandom_output_path, "bw") as semirandom_file:
        for _ in tqdm(range(output_bytes)):
            byte = np.random.randint(256)
            random_file.write(bytes([byte]))
            semirandom_file.write(bytes([byte & mask]))


## Fuentes estadísticas

### Cálculo de la entropía estadística de los archivos generados en la simulación de compresión (las distribuciones tienen que ser las mismas para que coincidan los resultados)

In [ ]:
symbols = ["0", "1"]
pmf = np.array([0.1, 0.9])
memory_pmf = np.array([[0.1, 0.9], [0.3, 0.7], [0.4, 0.6], [0.8, 0.2]])
# memory_pmf = np.array([[0.75, 0.25], [0.95, 0.05], [0, 1], [0.5, 0.5]])  # distribución alternativa para probar
output_size = 8000*2**10

source = information.Source((symbols, pmf))
memory_source = information.MemorySource((symbols, memory_pmf))
unconditional_source = memory_source.unconditional_source()

current_dir_path = Path.cwd()
input_dir_path = current_dir_path.parent.joinpath("output")
source_input_path = input_dir_path.joinpath("compression_sim_source_output")
memory_source_input_path = input_dir_path.joinpath("compression_sim_memory_source_output")
unconditional_source_input_path = input_dir_path.joinpath("compression_sim_unconditional_source_output")

estimated_pmf = np.zeros(pmf.shape)
n_bits = 0
with open(source_input_path, "br") as source_input_file:
    for _ in tqdm(range(output_size)):
        byte = source_input_file.read(1)
        byte_string = f"{int.from_bytes(byte):08b}"
        for bit in byte_string:
            symbol_index = 0 if bit == "0" else 1
            estimated_pmf[symbol_index] += 1
            n_bits += 1
            
    estimated_pmf /= n_bits
    estimated_source = information.Source((symbols, estimated_pmf))
    estimated_entropy = estimated_source.entropy()
    print("Fuente sin memoria")
    print(source)
    print(f"Entropía de la fuente: {source.entropy():.3g} bits")
    print("Estimación")
    print(estimated_source)
    print(f"Entropía estimada de la fuente: {estimated_entropy:.3g}")

estimated_pmf = np.zeros(memory_pmf.shape)
n_bits = 0
current_state = ""
with open(memory_source_input_path, "br") as memory_source_input_file:
    for _ in tqdm(range(output_size)):
        byte = memory_source_input_file.read(1)
        byte_string = f"{int.from_bytes(byte):08b}"
        for bit in byte_string:
            if len(current_state) < memory_source.memory:
                current_state += bit
            else:
                state_index = memory_source.state_to_index(current_state)
                symbol_index = 0 if bit == "0" else 1
                estimated_pmf[state_index, symbol_index] += 1
                n_bits += 1
                current_state += bit
                current_state = current_state[-memory_source.memory:]
            
    estimated_pmf /= n_bits
    estimated_source = information.MemorySource((symbols, estimated_pmf))
    estimated_entropy = estimated_source.entropy()
    print("Fuente con memoria")
    print(memory_source)
    print(f"Entropía de la fuente con memoria: {memory_source.entropy():.3g} bits")
    print("Fuente estimada")
    print(estimated_source)
    print(f"Entropía estimada de la fuente con memoria: {estimated_entropy:.3g}")

estimated_pmf = np.zeros(pmf.shape)
n_bits = 0
with open(memory_source_input_path, "br") as memory_source_input_file:
    for _ in tqdm(range(output_size)):
        byte = memory_source_input_file.read(1)
        byte_string = f"{int.from_bytes(byte):08b}"
        for bit in byte_string:
            symbol_index = 0 if bit == "0" else 1
            estimated_pmf[symbol_index] += 1
            n_bits += 1
            
    estimated_pmf /= n_bits
    estimated_source = information.Source((symbols, estimated_pmf))
    estimated_entropy = estimated_source.entropy()
    print("Fuente con memoria estimada como si no tuviese memoria")
    print(estimated_source)
    print(f"Entropía estimada de la fuente: {estimated_entropy:.3g}")

estimated_pmf = np.zeros(pmf.shape)
n_bits = 0
with open(unconditional_source_input_path, "br") as unconditional_source_input_file:
    for _ in tqdm(range(output_size)):
        byte = unconditional_source_input_file.read(1)
        byte_string = f"{int.from_bytes(byte):08b}"
        for bit in byte_string:
            symbol_index = 0 if bit == "0" else 1
            estimated_pmf[symbol_index] += 1
            n_bits += 1
            
    estimated_pmf /= n_bits
    estimated_source = information.Source((symbols, estimated_pmf))
    estimated_entropy = estimated_source.entropy()
    print("Fuente afín")
    print(unconditional_source)
    print(f"Entropía de la fuente afín: {unconditional_source.entropy():.3g} bits")
    print("Fuente estimada")
    print(estimated_source)
    print(f"Entropía estimada de la fuente afín: {estimated_entropy:.3g}")